# Latency Micro-Benchmark on Cloud GPU

Times the two-token, eight-token, and string-parse readouts on 50 POPE
questions and writes `experiments/latency_microbench.json`. Runs in
under 5 minutes on a T4. Feeds the latency paragraph in Section 9 of
the paper.

In [ ]:
MODEL_PATH = 'llava-hf/llava-1.5-7b-hf'
SPLIT = 'adversarial'
SAMPLES = 50
REPO_URL = 'https://github.com/Kesav2k04/ugaa-research.git'
REPO_BRANCH = 'ugaa-v2'

import os, sys, pathlib, subprocess
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
WORK = pathlib.Path('/kaggle/working') if IS_KAGGLE else (pathlib.Path('/content/work') if IS_COLAB else pathlib.Path.cwd() / 'cloud_run')
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess, sys
def pip(*a):
    return subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *a])
pip('transformers==4.40.1', 'accelerate>=0.28,<0.40', 'bitsandbytes>=0.43,<0.45')
pip('sentencepiece>=0.2.0', 'protobuf>=3.20,<5.0', 'safetensors')
pip('pandas', 'pyarrow', 'requests', 'Pillow')

In [ ]:
REPO_DIR = WORK / 'ugaa-research'
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth=1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--rebase'])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, 'scripts/download_pope_full.py', '--splits', SPLIT])

In [ ]:
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache/transformers'
os.makedirs('/tmp/hf_cache', exist_ok=True)
cmd = [
    sys.executable, 'analysis/latency_microbench.py',
    '--model-path', MODEL_PATH,
    '--data-dir', 'datasets/pope',
    '--split', SPLIT,
    '--samples', str(SAMPLES),
    '--device', 'cuda',
    '--cache-dir', '/tmp/hf_cache',
    '--output', 'experiments/latency_microbench.json',
]
print(' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
import json, shutil
with open('experiments/latency_microbench.json') as f:
    print(json.dumps(json.load(f), indent=2))
DEST = pathlib.Path('/kaggle/working/outputs') if IS_KAGGLE else (WORK / 'outputs')
DEST.mkdir(parents=True, exist_ok=True)
shutil.copy2('experiments/latency_microbench.json', DEST / 'latency_microbench.json')